# Motion Energy Distribution Analysis

This notebook analyzes the distribution of motion energy ($E_{acc}$, $E_{gyr}$) over the 1.5-second recording window across the collected training datasets. 

### Objectives:
1. **Global Analysis:** Aggregate all gesture records to see overall framing and stillness quality.
2. **Per-Gesture Analysis:** Inspect individual gesture curves to pinpoint dynamic patterns.
3. **Quality Ranking:** Calculate quantitative metrics (centering error, boundary stillness) to identify which gestures have the highest quality and which ones represent the worst training data (leakage or misalignment).

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Setup plotting backend
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

# Configuration
DATA_DIR = Path("../data")
SAMPLING_RATE_HZ = 100
WINDOW_SIZE_S = 1.5
TARGET_LENGTH = int(SAMPLING_RATE_HZ * WINDOW_SIZE_S)  # 150 samples

print(f"Target folder: {DATA_DIR.resolve()}")
print(f"Target dimensions: {TARGET_LENGTH} samples at {SAMPLING_RATE_HZ} Hz")

## Mathematical Definition of Motion Energy

For each time step $t$ in a gesture file, the motion energy is defined as the magnitude of the sensor vectors:

- **Accelerometer Motion Energy** ($E_{acc}$):
  $$E_{acc}(t) = \sqrt{a_x(t)^2 + a_y(t)^2 + a_z(t)^2}$$

- **Gyroscope Motion Energy (Rotational Energy)** ($E_{gyr}$):
  $$E_{gyr}(t) = \sqrt{\omega_x(t)^2 + \omega_y(t)^2 + \omega_z(t)^2}$$

We will compute these magnitudes for both **IMU1 (Wrist)** and **IMU2 (Finger)**.

In [ ]:
def calculate_sample_energy(csv_path: Path):
    """Loads a recording CSV and returns energy profiles normalized to target length."""
    df = pd.read_csv(csv_path)
    
    # Standardize length to exactly 150 samples
    if len(df) < TARGET_LENGTH:
        pad_size = TARGET_LENGTH - len(df)
        last_row = df.iloc[-1:]
        df = pd.concat([df, pd.concat([last_row] * pad_size, ignore_index=True)], ignore_index=True)
    elif len(df) > TARGET_LENGTH:
        df = df.iloc[:TARGET_LENGTH]
        
    profiles = {}
    # IMU1 (wrist)
    profiles['IMU1_E_acc'] = np.sqrt(df['IMU1_accX']**2 + df['IMU1_accY']**2 + df['IMU1_accZ']**2)
    profiles['IMU1_E_gyr'] = np.sqrt(df['IMU1_gyrX']**2 + df['IMU1_gyrY']**2 + df['IMU1_gyrZ']**2)
    
    # IMU2 (finger)
    profiles['IMU2_E_acc'] = np.sqrt(df['IMU2_accX']**2 + df['IMU2_accY']**2 + df['IMU2_accZ']**2)
    profiles['IMU2_E_gyr'] = np.sqrt(df['IMU2_gyrX']**2 + df['IMU2_gyrY']**2 + df['IMU2_gyrZ']**2)
    
    return profiles

## Processing Gesture Datasets

In [ ]:
gestures = [d.name for d in DATA_DIR.iterdir() if d.is_dir()]
print(f"Detected gesture categories: {gestures}")

gesture_profiles = {}
all_imu1_acc = []
all_imu1_gyr = []
all_imu2_acc = []
all_imu2_gyr = []

for gesture in gestures:
    # Find all CSV files recursively, excluding calibration.csv and energy_distribution.csv
    files = [Path(f) for f in DATA_DIR.glob(f"{gesture}/**/*.csv") if Path(f).name not in ("calibration.csv", "energy_distribution.csv")]
    if not files:
        print(f"  -> '{gesture}' is empty or contains no valid sample files.")
        continue
        
    print(f"Processing '{gesture}' ({len(files)} files)...")
    
    run_data = {
        'IMU1_acc': [], 'IMU1_gyr': [],
        'IMU2_acc': [], 'IMU2_gyr': []
    }
    
    for f in files:
        try:
            p = calculate_sample_energy(f)
            run_data['IMU1_acc'].append(p['IMU1_E_acc'])
            run_data['IMU1_gyr'].append(p['IMU1_E_gyr'])
            run_data['IMU2_acc'].append(p['IMU2_E_acc'])
            run_data['IMU2_gyr'].append(p['IMU2_E_gyr'])
            
            # Add to global aggregates
            all_imu1_acc.append(p['IMU1_E_acc'])
            all_imu1_gyr.append(p['IMU1_E_gyr'])
            all_imu2_acc.append(p['IMU2_E_acc'])
            all_imu2_gyr.append(p['IMU2_E_gyr'])
        except Exception as e:
            print(f"    Error parsing {f.name}: {e}")
            
    if run_data['IMU1_acc']:
        gesture_profiles[gesture] = {
            'IMU1_acc_runs': np.array(run_data['IMU1_acc']),
            'IMU1_gyr_runs': np.array(run_data['IMU1_gyr']),
            'IMU2_acc_runs': np.array(run_data['IMU2_acc']),
            'IMU2_gyr_runs': np.array(run_data['IMU2_gyr']),
            'count': len(run_data['IMU1_acc'])
        }
        print(f"    Successfully aggregated {gesture_profiles[gesture]['count']} samples.")

## 1. Global Analysis

We analyze the global motion energy profiles averaged across all gesture categories in the dataset.

In [ ]:
if all_imu1_acc:
    all_imu1_acc = np.array(all_imu1_acc)
    all_imu1_gyr = np.array(all_imu1_gyr)
    all_imu2_acc = np.array(all_imu2_acc)
    all_imu2_gyr = np.array(all_imu2_gyr)
    
    t = np.linspace(0, WINDOW_SIZE_S, TARGET_LENGTH)
    fig, axs = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
    fig.suptitle(f"Gesamt-Bewegungsenergie-Verteilung (Gesamter Datensatz: {len(all_imu1_acc)} Samples)", fontsize=16, fontweight='bold')
    
    # --- IMU1 Accelerometer (Wrist) ---
    mean_acc1 = np.mean(all_imu1_acc, axis=0)
    std_acc1 = np.std(all_imu1_acc, axis=0)
    axs[0, 0].plot(t, mean_acc1, color='#1f77b4', label='Mean')
    axs[0, 0].fill_between(t, mean_acc1 - std_acc1, mean_acc1 + std_acc1, color='#1f77b4', alpha=0.15)
    axs[0, 0].set_title("IMU1 (Wrist) Accelerometer Magnitude")
    axs[0, 0].set_ylabel("Acceleration (g)")
    axs[0, 0].grid(True, linestyle='--', alpha=0.7)
    axs[0, 0].legend(loc='upper right')
    
    # --- IMU1 Gyroscope (Wrist) ---
    mean_gyr1 = np.mean(all_imu1_gyr, axis=0)
    std_gyr1 = np.std(all_imu1_gyr, axis=0)
    axs[1, 0].plot(t, mean_gyr1, color='#ff7f0e', label='Mean')
    axs[1, 0].fill_between(t, mean_gyr1 - std_gyr1, mean_gyr1 + std_gyr1, color='#ff7f0e', alpha=0.15)
    axs[1, 0].set_title("IMU1 (Wrist) Gyroscope Magnitude")
    axs[1, 0].set_ylabel("Angular Velocity (dps)")
    axs[1, 0].set_xlabel("Time (seconds)")
    axs[1, 0].grid(True, linestyle='--', alpha=0.7)
    axs[1, 0].legend(loc='upper right')
    
    # --- IMU2 Accelerometer (Finger) ---
    mean_acc2 = np.mean(all_imu2_acc, axis=0)
    std_acc2 = np.std(all_imu2_acc, axis=0)
    axs[0, 1].plot(t, mean_acc2, color='#2ca02c', label='Mean')
    axs[0, 1].fill_between(t, mean_acc2 - std_acc2, mean_acc2 + std_acc2, color='#2ca02c', alpha=0.15)
    axs[0, 1].set_title("IMU2 (Finger) Accelerometer Magnitude")
    axs[0, 1].grid(True, linestyle='--', alpha=0.7)
    axs[0, 1].legend(loc='upper right')
    
    # --- IMU2 Gyroscope (Finger) ---
    mean_gyr2 = np.mean(all_imu2_gyr, axis=0)
    std_gyr2 = np.std(all_imu2_gyr, axis=0)
    axs[1, 1].plot(t, mean_gyr2, color='#d62728', label='Mean')
    axs[1, 1].fill_between(t, mean_gyr2 - std_gyr2, mean_gyr2 + std_gyr2, color='#d62728', alpha=0.15)
    axs[1, 1].set_title("IMU2 (Finger) Gyroscope Magnitude")
    axs[1, 1].set_xlabel("Time (seconds)")
    axs[1, 1].grid(True, linestyle='--', alpha=0.7)
    axs[1, 1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()
else:
    print("No aggregated runs to analyze globally.")

## 2. Per-Gesture Analysis

In [ ]:
t = np.linspace(0, WINDOW_SIZE_S, TARGET_LENGTH)

for gesture, data in gesture_profiles.items():
    fig, axs = plt.subplots(2, 2, figsize=(16, 9), sharex=True)
    fig.suptitle(f"Motion Energy Profiles: Geste '{gesture}' ({data['count']} Samples)", fontsize=16, fontweight='bold')
    
    # --- IMU1 Accelerometer ---
    mean_acc1 = np.mean(data['IMU1_acc_runs'], axis=0)
    std_acc1 = np.std(data['IMU1_acc_runs'], axis=0)
    axs[0, 0].plot(t, mean_acc1, color='#1f77b4', label='Mean')
    axs[0, 0].fill_between(t, mean_acc1 - std_acc1, mean_acc1 + std_acc1, color='#1f77b4', alpha=0.15)
    axs[0, 0].set_title("IMU1 (Wrist) Accelerometer Magnitude")
    axs[0, 0].set_ylabel("Acceleration (g)")
    axs[0, 0].grid(True, linestyle='--', alpha=0.7)
    axs[0, 0].legend(loc='upper right')
    
    # --- IMU1 Gyroscope ---
    mean_gyr1 = np.mean(data['IMU1_gyr_runs'], axis=0)
    std_gyr1 = np.std(data['IMU1_gyr_runs'], axis=0)
    axs[1, 0].plot(t, mean_gyr1, color='#ff7f0e', label='Mean')
    axs[1, 0].fill_between(t, mean_gyr1 - std_gyr1, mean_gyr1 + std_gyr1, color='#ff7f0e', alpha=0.15)
    axs[1, 0].set_title("IMU1 (Wrist) Gyroscope Magnitude")
    axs[1, 0].set_ylabel("Angular Velocity (dps)")
    axs[1, 0].set_xlabel("Time (seconds)")
    axs[1, 0].grid(True, linestyle='--', alpha=0.7)
    axs[1, 0].legend(loc='upper right')
    
    # --- IMU2 Accelerometer ---
    mean_acc2 = np.mean(data['IMU2_acc_runs'], axis=0)
    std_acc2 = np.std(data['IMU2_acc_runs'], axis=0)
    axs[0, 1].plot(t, mean_acc2, color='#2ca02c', label='Mean')
    axs[0, 1].fill_between(t, mean_acc2 - std_acc2, mean_acc2 + std_acc2, color='#2ca02c', alpha=0.15)
    axs[0, 1].set_title("IMU2 (Finger) Accelerometer Magnitude")
    axs[0, 1].grid(True, linestyle='--', alpha=0.7)
    axs[0, 1].legend(loc='upper right')
    
    # --- IMU2 Gyroscope ---
    mean_gyr2 = np.mean(data['IMU2_gyr_runs'], axis=0)
    std_gyr2 = np.std(data['IMU2_gyr_runs'], axis=0)
    axs[1, 1].plot(t, mean_gyr2, color='#d62728', label='Mean')
    axs[1, 1].fill_between(t, mean_gyr2 - std_gyr2, mean_gyr2 + std_gyr2, color='#d62728', alpha=0.15)
    axs[1, 1].set_title("IMU2 (Finger) Gyroscope Magnitude")
    axs[1, 1].set_xlabel("Time (seconds)")
    axs[1, 1].grid(True, linestyle='--', alpha=0.7)
    axs[1, 1].legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

## 3. Gesture Data Quality Score

To analyze where the training data quality is highest and which gestures represent the worst/messiest training data, we define three mathematical quality metrics:

1. **Start Stillness (dps):** Mean wrist gyroscope magnitude ($E_{gyr}$) in the first 20 samples (0.0 to 0.2s). Lower is better ($<8$ dps is excellent).
2. **End Stillness (dps):** Mean wrist gyroscope magnitude ($E_{gyr}$) in the last 20 samples (1.3 to 1.5s). Lower is better ($<8$ dps is excellent).
3. **Centering Error (seconds):** The absolute offset of the peak rotation energy from the exact center (0.75s). Lower is better ($<0.15$s is excellent).

Gestures with high boundary stillness values suffer from **motion leakage** (i.e. starting the gesture too early or moving arm after ending). Gestures with high centering error suffer from **unaligned peak windows**.

In [ ]:
quality_metrics = []

for gesture, data in gesture_profiles.items():
    # Calculate mean curves
    mean_gyr1 = np.mean(data['IMU1_gyr_runs'], axis=0)
    
    # Stillness metric (first and last 20 samples)
    start_stillness = np.mean(mean_gyr1[:20])
    end_stillness = np.mean(mean_gyr1[-20:])
    
    # Peak centering
    peak_idx = np.argmax(mean_gyr1)
    peak_time = peak_idx * (WINDOW_SIZE_S / TARGET_LENGTH)
    centering_error = abs(peak_time - 0.75)
    
    # Classify Quality
    if start_stillness < 8.0 and end_stillness < 8.0 and centering_error < 0.12:
        rating = "🟢 Excellent"
    elif start_stillness < 15.0 and end_stillness < 15.0 and centering_error < 0.22:
        rating = "🟡 Good"
    else:
        rating = "🔴 Needs Improvement"
        
    quality_metrics.append({
        'Geste': gesture,
        'Anzahl Samples': data['count'],
        'Start Stillness (dps)': round(start_stillness, 2),
        'End Stillness (dps)': round(end_stillness, 2),
        'Peak Time (s)': round(peak_time, 2),
        'Centering Error (s)': round(centering_error, 2),
        'Qualitäts-Rating': rating
    })

df_quality = pd.DataFrame(quality_metrics)
# Rank gestures: Excellent / Good / Needs Improvement based on a composite penalty
df_quality['penalty'] = df_quality['Centering Error (s)'] * 50 + df_quality['Start Stillness (dps)'] + df_quality['End Stillness (dps)']
df_quality = df_quality.sort_values(by='penalty').drop(columns=['penalty'])

print("=========================================================================")
print("            Gesture Training Data Quality Score Comparison               ")
print("=========================================================================")
display(df_quality)

## 4. Interpretation and Diagnosis

Review the ranking table above to diagnose dataset improvements:

- **High-Quality Gestures (🟢 Excellent):** The boundaries are stationary (gyroscope $<8$ dps), and the peak is centered in the window. These samples will yield excellent CNN classifier representation.
- **Poor-Quality Gestures (🔴 Needs Improvement):**
  - If **Start/End Stillness** is high: The user is moving their arm during the designated stillness phases. *Fix: Hold the wrist/hand completely still for 0.2s before initiating the gesture and immediately after finishing it.*
  - If **Centering Error** is high: The user is rushing the gesture (resulting in a left-shifted peak) or delaying it (right-shifted peak). *Fix: Attempt to perform the gesture peak speed exactly in the middle (0.75s) of the 1.5-second recording phase.*